In [1]:
!pip install datasets==4.8.4 emoji==2.15.0 numpy==2.4.4 pandas==3.0.2 scikit-learn==1.8.0 torch==2.11.0 transformers==5.5.4 accelerate==1.13.0

In [2]:
import pandas as pd
import numpy as np

# For machine learning tools and evaluation
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, balanced_accuracy_score


from sklearn.model_selection import train_test_split

import emoji
import re
import datasets
#from transformers import TrainingArguments, Trainer
#from transformers import AutoTokenizer, AutoModelForSequenceClassification
import warnings
from scipy.stats import chisquare

import torch

In [3]:
df_inference =  pd.read_csv("df_inference.csv")

# Convert 'label_text_pred' to binary values
df_inference['label_pred'] = df_inference['label_text_pred'].apply(lambda x: 1 if x == 'LABEL_1' else 0)

#Split data
joint_data_2020 = df_inference[df_inference['year'] == 2020]
joint_data_2021 = df_inference[df_inference['year'] != 2020]

Calculate recall and precision for overall dataset and datasets split on year

In [4]:
# Calculate precision and recall overall (to test if this gets me the same results as above)
precision = precision_recall_fscore_support(y_true=df_inference['labels'], y_pred=df_inference['label_pred'], average='macro')
#recall = recall_score(df_inference['labels'], df_inference['label_pred'], average= 'binary')
print(f'Precision: {precision}')
#print(f'Recall: {recall}')

Precision: (0.7537855023314963, 0.7516646489104116, 0.7527142712413886, None)


In [5]:
# Calculate precision and recall (2020)
precision_recall_f = precision_recall_fscore_support(y_true=joint_data_2020['labels'], y_pred=joint_data_2020['label_pred'], average='binary')
precision_recall_f_macro = precision_recall_fscore_support(y_true=joint_data_2020['labels'], y_pred=joint_data_2020['label_pred'], average='macro')
accuracy = accuracy_score(y_true=joint_data_2020['labels'], y_pred=joint_data_2020['label_pred'])
print(f'Precision_recall_fscore: {precision_recall_f}')
print(f'Precision_recall_fscore_macro: {precision_recall_f_macro}')
print(f'Accuracy: {accuracy}')

Precision_recall_fscore: (0.5666666666666667, 0.4857142857142857, 0.5230769230769231, None)
Precision_recall_fscore_macro: (0.7383333333333333, 0.7095238095238096, 0.7222979552093476, None)
Accuracy: 0.8652173913043478


In [6]:
# Calculate precision and recall (after 2020)
precision_recall_f = precision_recall_fscore_support(y_true=joint_data_2021['labels'], y_pred=joint_data_2021['label_pred'], average='binary')
precision_recall_f_macro = precision_recall_fscore_support(y_true=joint_data_2021['labels'], y_pred=joint_data_2021['label_pred'], average='macro')
accuracy = accuracy_score(y_true=joint_data_2021['labels'], y_pred=joint_data_2021['label_pred'])
print(f'Precision_recall_fscore: {precision_recall_f}')
print(f'Precision_recall_fscore_macro: {precision_recall_f_macro}')
print(f'Accuracy: {accuracy}')

Precision_recall_fscore: (0.6347826086956522, 0.6822429906542056, 0.6576576576576577, None)
Precision_recall_fscore_macro: (0.7553475087273882, 0.7666534102207199, 0.7604835050878216, None)
Accuracy: 0.8046272493573264


Calculate metrics overall and per year for accurate information Tweets


In [7]:
precision = precision_recall_fscore_support(y_true=df_inference['labels'], y_pred=df_inference['label_pred'], average='binary', pos_label =0)
print(f'Precision, recall, fscore: {precision}')

Precision, recall, fscore: (0.8961424332344213, 0.8988095238095238, 0.8974739970282318, None)


In [8]:
# Calculate precision and recall (2020)
precision_recall_f = precision_recall_fscore_support(y_true=joint_data_2020['labels'], y_pred=joint_data_2020['label_pred'], average='binary', pos_label =0)
print(f'Precision_recall_fscore: {precision_recall_f}')

Precision_recall_fscore: (0.91, 0.9333333333333333, 0.9215189873417722, None)


In [9]:
# Calculate precision and recall (after 2020)
precision_recall_f = precision_recall_fscore_support(y_true=joint_data_2021['labels'], y_pred=joint_data_2021['label_pred'], average='binary', pos_label =0)
print(f'Precision_recall_fscore: {precision_recall_f}')

Precision_recall_fscore: (0.8759124087591241, 0.851063829787234, 0.8633093525179856, None)


$\chi^2$ test

In [10]:
#merge 2021 and 2022 in data inference dataset
df_inference['year'] = df_inference['year'].apply(lambda x: 2020 if x == 2020 else 2021)

In [11]:
#calculate function for whether labels were correctly predicted
#Make recall variable
#1 means model correctly predicted label
# 0 means did not correctly predict label
def num(row):
    if row['labels'] == row['label_pred']:
        return 1
    else :
        return 0

df_inference['correct'] = df_inference.apply(num, axis =1)

Recall misinformation

In [12]:

#Select only misinformation variable (to calculate significance for recall misinformation variable, not accuracy)
df_inf_mis = df_inference[df_inference["labels"]==1]


#make table of number of tweets in each group
df_inf_mis.groupby(['correct', 'year']).size()

correct  year
0        2020    36
         2021    34
1        2020    34
         2021    73
dtype: int64

In [13]:
#Test difference between 2020 and 2021
chisquare([36, 34, 34, 73])

Power_divergenceResult(statistic=np.float64(24.966101694915253), pvalue=np.float64(1.5694545833270573e-05))

Recall accurate information

In [14]:
#Select only accurate information variable (to calculate significance for recall accurate information)
df_inf_mis = df_inference[df_inference["labels"]==0]


#make table of number of tweets in each group
df_inf_mis.groupby(['correct', 'year']).size()

correct  year
0        2020     26
         2021     42
1        2020    364
         2021    240
dtype: int64

In [15]:
#Test difference between 2020 and 2021
chisquare([26, 42, 364, 240])

Power_divergenceResult(statistic=np.float64(474.047619047619), pvalue=np.float64(2.0073899710508232e-102))

Accuracy

In [16]:
df_inference.groupby(['correct','year']).size()

correct  year
0        2020     62
         2021     76
1        2020    398
         2021    313
dtype: int64

In [17]:
chisquare([62, 76, 398, 313])

Power_divergenceResult(statistic=np.float64(404.2061248527679), pvalue=np.float64(2.716859614077706e-87))

Precision misinformation

In [18]:
#Select only predicted misinformation (to calculate significance for precision misinformation label)
df_inf_mis = df_inference[df_inference["label_pred"]==1]
#make table of number of tweets in each group
df_inf_mis.groupby(['correct', 'year']).size()

correct  year
0        2020    26
         2021    42
1        2020    34
         2021    73
dtype: int64

In [19]:
chisquare([26,42,34,73])

Power_divergenceResult(statistic=np.float64(29.0), pvalue=np.float64(2.2394290022533747e-06))

precision accurate information

In [20]:
#Select only predicted accurate information (to calculate significance for precision accurate information)
df_inf_mis = df_inference[df_inference["label_pred"]==0]
#make table of number of tweets in each group
df_inf_mis.groupby(['correct', 'year']).size()

correct  year
0        2020     36
         2021     34
1        2020    364
         2021    240
dtype: int64

In [21]:
chisquare([36,34,364,240])

Power_divergenceResult(statistic=np.float64(468.7181008902077), pvalue=np.float64(2.8673264612759256e-101))